In [0]:
from pathlib import Path
import pandas as pd
import numpy as np

In [0]:
%sql
--DROP DATABASE IF EXISTS workspace.bronze CASCADE; 
drop table if exists workspace.silver.tbl_representantes;
drop table if exists workspace.silver.tbl_productos;
drop table if exists workspace.silver.tbl_ventas_detalle;

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.silver
COMMENT 'Capa Silver procesados'


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.tbl_representantes (
  `Representante` STRING,
  `Ciudad` STRING, 
  `Fotografía` DOUBLE
)

""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.tbl_productos (
    `CódigoProducto` STRING,
    `Descripción` STRING,
    `Precio de venta` DOUBLE,
    `Costo de venta` BIGINT,
    `Almacen` BIGINT,
    `Vendidos` BIGINT
)USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name'
)

""")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.tbl_ventas_detalle (
  `Fecha` TIMESTAMP,
  `Representante` STRING,
  `CódigoProducto` STRING,
  `Unidades` BIGINT
)

""")

In [0]:
# Limpia nombres de columnas para estandarizarlos:
# - quita espacios al inicio/fin
# - convierte a minúsculas
# - reemplaza espacios, puntos y guiones por "_"
def clean_column_name(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace(".", "_").replace("-", "_").replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")

# Convierte valores complejos a formatos comparables:
# - list  -> tuple
# - ndarray -> tuple
# - dict -> tuple ordenada de pares (key, value)
def normalize_cell(value):
    if isinstance(value, list):
        return tuple(value)
    if isinstance(value, np.ndarray):
        return tuple(value.tolist())
    if isinstance(value, dict):
        return tuple(sorted(value.items()))
    return value

# Si el valor es lista/tupla, lo convierte a string separado por comas.
# Si no, lo deja igual.
# Se usa para columnas como géneros o días de programación.
def join_if_sequence(value):
    if isinstance(value, (list, tuple)):
        return ",".join(str(item) for item in value)
    return value

In [0]:
# Leer bronze y pasar a pandas
df_spark = spark.table("workspace.bronze.tbl_representantes")
df = df_spark.toPandas()

In [0]:
# Limpiar nombres de columnas
df.columns = [clean_column_name(col) for col in df.columns]

In [0]:
df.columns

In [0]:
# 5) Normalizar tipos complejos para poder deduplicar
df = df.apply(lambda col: col.map(normalize_cell))

In [0]:
# 6) Quitar duplicados
df = df.drop_duplicates()

In [0]:
df_spark = spark.createDataFrame(df)


In [0]:
df_spark

In [0]:
# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tbl_representantes")

In [0]:
df_spark = df_spark.withColumnRenamed(
    "Fotografía",
    "fotografia"
)

df_spark = df_spark.withColumnRenamed(
    "CódigoProducto",
    "codigoproducto")
df_spark = df_spark.withColumnRenamed(
    "Descripción",
    "descripcion")
df_spark = df_spark.withColumnRenamed(
    "Costo de venta",
    "costodeventa")
df_spark = df_spark.withColumnRenamed(
    "Precio de venta",
    "precioventa")
    
    

In [0]:
df_spark = df_spark.withColumnRenamed(
    "CódigoProducto",
    "codigoproducto")

In [0]:
%sql
select *from workspace.bronze.tbl_ventas_detalle a
 left join workspace.bronze.tbl_representantes b 
    on(upper(trim(b.representante))=upper(trim(a.representante)))
where
    b.representante is null

;


--select *from workspace.bronze.tbl_representantes b  where upper(b.Representante) like '%TAPIA%CAST%'; --Valeria Tapia Castro